## Demo 03 - Regression Analysis

Up to this point, we've performed some basic descriptive analytics, mostly looking at counts and averages.  We can also use regression analysis to understand our data better.

In [ ]:
from sqlalchemy import create_engine
import pandas as pd
import tomllib

with open("../config.toml", "rb") as f:
    config = tomllib.load(f)
server = config["database"]["server"]
database = config["database"]["name"]
username = config["database"]["username"]
password = config["database"]["password"]
engine = create_engine(f"mssql+pyodbc://{username}:{password}@{server}/{database}?driver=ODBC+Driver+18+for+SQL+Server&TrustServerCertificate=yes")


## Regressing Expenditures against Bus Counts

To start with, we will perform a fairly broad regression, looking at data by month but aggregated at a fairly broad grain, so there isn't a huge amount of aggregation.

We will use a query similar to the one we looked at for correlation. This query includes two different types of variables: <strong>continuous</strong> and <strong>categorical</strong>. Continuous variables are numeric values which can change over a range of values, such as average road conditions or bus model quality. Meanwhile, categorical variables represent a fixed set of possible values, such as employee names or model qualities.

In [ ]:
raw_data = pd.read_sql("""SELECT
    c.CalendarYear,
    c.CalendarMonth,
    b.AverageRoadConditions,
    b.BusModelID,
    bm.BusModel,
    bm.ModelQuality,
    DATEDIFF(YEAR, b.DateFirstInService, li.LineItemDate) AS NumberOfYearsInService,
    ec.ExpenseCategory,
    v.VendorName,
    CONCAT(e.FirstName, ' ', e.LastName) AS EmployeeName,
    SUM(li.Amount) AS TotalAmount,
    COUNT(*) AS NumberOfLineItems
FROM dbo.LineItem li
    INNER JOIN dbo.Calendar c
        ON li.LineItemDate = c.Date
    INNER JOIN dbo.Bus b
        ON li.BusID = b.BusID
    INNER JOIN dbo.BusModel bm
        ON b.BusModelID = bm.BusModelID
    INNER JOIN dbo.ExpenseCategory ec
        ON li.ExpenseCategoryID = ec.ExpenseCategoryID
    INNER JOIN dbo.Vendor v
        ON li.VendorID = v.VendorID
    INNER JOIN dbo.Employee e
        ON li.EmployeeID = e.EmployeeID
GROUP BY
    c.CalendarYear,
    c.CalendarMonth,
    b.AverageRoadConditions,
    b.BusModelID,
    bm.BusModel,
    bm.ModelQuality,
    DATEDIFF(YEAR, b.DateFirstInService, li.LineItemDate),
    ec.ExpenseCategory,
    v.VendorName,
    CONCAT(e.FirstName, ' ', e.LastName)
ORDER BY NEWID();""", engine)

Our first attempt will be a basic linear regression. This will give us a baseline and starting point for more detailed analysis.

In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, root_mean_squared_error
import statsmodels.api as sm

# Build the independent variables
reg1_x = raw_data[["CalendarYear", "CalendarMonth", "AverageRoadConditions", "BusModel", "ModelQuality", "NumberOfYearsInService", "ExpenseCategory", "VendorName", "EmployeeName"]]
reg1_x_onehot = pd.get_dummies(reg1_x)

# Capture the dependent variable
reg1_y = raw_data["TotalAmount"]

# Split out into training and test data sets
reg1_x_train = reg1_x_onehot[:-30000]
reg1_x_test = reg1_x_onehot[-30000:]

reg1_y_train = reg1_y[:-30000]
reg1_y_test = reg1_y[-30000:]

regr = LinearRegression()
model = regr.fit(reg1_x_train, reg1_y_train)
sm_model = sm.OLS(reg1_y_train, reg1_x_train.astype(float)).fit()


In [ ]:
sm_model.summary()

In [ ]:
print('Training data set Coefficient of Determination', model.score(reg1_x_train, reg1_y_train))
print('Test data set Coefficient of Determination', model.score(reg1_x_test, reg1_y_test))

reg1_y_pred = regr.predict(reg1_x_test)
print('Mean squared error: %.2f' % mean_squared_error(reg1_y_test, reg1_y_pred) )
print('Root mean squared error: %.2f' % root_mean_squared_error(reg1_y_test, reg1_y_pred) )

What this tells us is that our regression model explains approximately 55% of total variance in line item amounts. That's definitely not a bad start. Some things we can recognize from this first attempt are:

1. It looks like there are two bands of employees: one in a -160 grouping (Bonden, Oakes, Graham, J. Aubrey, Martin, S. Aubrey, Maturin, Babbington) and one in a -150 grouping (Villiers, Sweeting, Harte, Mowett). Given that the standard error for each is 20 USD, that might not be particularly significant.
2. Model quality is definitely significant: a full point in model quality would lower line item price by a large amount. Our actual model quality range is smaller than a full point, but that's still a big difference.
3. Interestingly, average road conditions don't seem to affect line item price very much.
4. Prices do tend to go up about 1 USD per year on average per line item. That might be inflation or it could be something else--remember that we did see a massive spike near the end of our observation period, so this might be reflective of it.
5. Many of these numbers look weird because we're starting calendar year at 2011, so the line item starts at `2011 * 1.0946` = 2201 USD. (Note that the weight might be different based on the seed of your run.) If we start calendar year at 0, by subtracting 2011 from each calendar year, the results won't change but the scales will be more human-understandable.

1. The mean squared error is pretty high.  If you take the square root of this, you'll get a number in USD which represents how far off we are.  There's a pretty wide range here, and considering that the mean amount in our data sample is 197 USD, it's not great.

## Regression: Pre-2019 Model

One of the things we noticed was that amounts (both prices and numbers of invoices) tended to change considerably between 2019 and 2021. We can see if there was a structural change in per-item amount by repeating this regression against data from 2011-2018 + 2022 and applying those results to the 2019-2021 data sets.

In [ ]:
raw_data_pre2019 = pd.read_sql("""SELECT
    c.CalendarYear,
    c.CalendarMonth,
    b.AverageRoadConditions,
    b.BusModelID,
    bm.BusModel,
    bm.ModelQuality,
    DATEDIFF(YEAR, b.DateFirstInService, li.LineItemDate) AS NumberOfYearsInService,
    ec.ExpenseCategory,
    v.VendorName,
    CONCAT(e.FirstName, ' ', e.LastName) AS EmployeeName,
    SUM(li.Amount) AS TotalAmount,
    COUNT(*) AS NumberOfLineItems
FROM dbo.LineItem li
    INNER JOIN dbo.Calendar c
        ON li.LineItemDate = c.Date
    INNER JOIN dbo.Bus b
        ON li.BusID = b.BusID
    INNER JOIN dbo.BusModel bm
        ON b.BusModelID = bm.BusModelID
    INNER JOIN dbo.ExpenseCategory ec
        ON li.ExpenseCategoryID = ec.ExpenseCategoryID
    INNER JOIN dbo.Vendor v
        ON li.VendorID = v.VendorID
    INNER JOIN dbo.Employee e
        ON li.EmployeeID = e.EmployeeID
WHERE
    c.CalendarYear BETWEEN 2011 AND 2018
    OR c.CalendarYear = 2022
GROUP BY
    c.CalendarYear,
    c.CalendarMonth,
    b.AverageRoadConditions,
    b.BusModelID,
    bm.BusModel,
    bm.ModelQuality,
    DATEDIFF(YEAR, b.DateFirstInService, li.LineItemDate),
    ec.ExpenseCategory,
    v.VendorName,
    CONCAT(e.FirstName, ' ', e.LastName)
ORDER BY NEWID();""", engine)

In [ ]:
# Build the independent variables
reg1p_x = raw_data_pre2019[["CalendarYear", "CalendarMonth", "AverageRoadConditions", "BusModel", "ModelQuality", "NumberOfYearsInService", "ExpenseCategory", "VendorName", "EmployeeName"]]
reg1p_x_onehot = pd.get_dummies(reg1p_x)

# Capture the dependent variable
reg1p_y = raw_data_pre2019["TotalAmount"]

# Split out into training and test data sets
# Using a smaller number for testing because of the reduced number of years.
reg1p_x_train = reg1p_x_onehot[:-20000]
reg1p_x_test = reg1p_x_onehot[-20000:]

reg1p_y_train = reg1p_y[:-20000]
reg1p_y_test = reg1p_y[-20000:]

regrp = LinearRegression()
modelp = regrp.fit(reg1p_x_train, reg1p_y_train)
sm_modelp = sm.OLS(reg1p_y_train, reg1p_x_train.astype(float)).fit()


In [ ]:
sm_modelp.summary()

In [ ]:
print('Training data set Coefficient of Determination', modelp.score(reg1p_x_train, reg1p_y_train))
print('Test data set Coefficient of Determination', modelp.score(reg1p_x_test, reg1p_y_test))

reg1p_y_pred = regrp.predict(reg1p_x_test)
print('Root mean squared error: %.2f' % root_mean_squared_error(reg1p_y_test, reg1p_y_pred) )

The coefficient of determination hasn't changed very much--it moved from 55% to 56%. One interesting finding is that employee appears not to be a relevant feature in this version of the model: all of the employees range between about 53-58 USD, so it's a good bit tighter than the entire model set. On the plus side, the mean squared error does decrease, and the root mean squared error is closer to 172 USD.

To get a feeling for how far off 2019-2021 is, we can make predictions using the "pre-2019" model and compare them to actuals from 2019-2021. We'll start out by getting the raw data for 2019-2021 as we did for other years. Then, we'll feed this data into the regression model.

In [ ]:
raw_data_2019 = pd.read_sql("""SELECT
    c.CalendarYear,
    c.CalendarMonth,
    b.AverageRoadConditions,
    b.BusModelID,
    bm.BusModel,
    bm.ModelQuality,
    DATEDIFF(YEAR, b.DateFirstInService, li.LineItemDate) AS NumberOfYearsInService,
    ec.ExpenseCategoryID,
    ec.ExpenseCategory,
    v.VendorID,
    v.VendorName,
    e.EmployeeID,
    CONCAT(e.FirstName, ' ', e.LastName) AS EmployeeName,
    SUM(li.Amount) AS TotalAmount,
    COUNT(*) AS NumberOfLineItems
FROM dbo.LineItem li
    INNER JOIN dbo.Calendar c
        ON li.LineItemDate = c.Date
    INNER JOIN dbo.Bus b
        ON li.BusID = b.BusID
    INNER JOIN dbo.BusModel bm
        ON b.BusModelID = bm.BusModelID
    INNER JOIN dbo.ExpenseCategory ec
        ON li.ExpenseCategoryID = ec.ExpenseCategoryID
    INNER JOIN dbo.Vendor v
        ON li.VendorID = v.VendorID
    INNER JOIN dbo.Employee e
        ON li.EmployeeID = e.EmployeeID
WHERE
    c.CalendarYear BETWEEN 2019 AND 2021
GROUP BY
    c.CalendarYear,
    c.CalendarMonth,
    b.AverageRoadConditions,
    b.BusModelID,
    bm.BusModel,
    bm.ModelQuality,
    DATEDIFF(YEAR, b.DateFirstInService, li.LineItemDate),
    ec.ExpenseCategoryID,
    ec.ExpenseCategory,
    v.VendorID,
    v.VendorName,
    e.EmployeeID,
    CONCAT(e.FirstName, ' ', e.LastName)
ORDER BY NEWID();""", engine)

In [ ]:
# Build the independent variables
reg1p2_x = raw_data_2019[["CalendarYear", "CalendarMonth", "AverageRoadConditions", "BusModel", "ModelQuality", "NumberOfYearsInService", "ExpenseCategory", "VendorName", "EmployeeName"]]
reg1p2_x_onehot = pd.get_dummies(reg1p2_x)

# Capture the dependent variable
reg1p2_y = raw_data_2019["TotalAmount"]

# We're using the entire data set here as we aren't training a model.
pred2019 = regrp.predict(reg1p2_x_onehot)
print('Root mean squared error: %.2f' % root_mean_squared_error(reg1p2_y, pred2019) )

In [ ]:
# Combine together the independent variables, the actual amount, and the predicted amount.
pred2019df = pd.concat([raw_data_2019, pd.DataFrame({'Prediction':pred2019})], axis=1)
pred2019df[["CalendarYear", "BusModelID", "ModelQuality", "ExpenseCategoryID", "VendorID", "EmployeeID", "TotalAmount", "Prediction"]]

Because our results aren't extremely accurate, there's a risk in focusing too much on them and expecting predictions to be close to total amounts. Let's see what it looks like in aggregate, however:

In [ ]:
print('Predicted amount: %.2f' % pred2019df["Prediction"].sum() )
print('Total amount: %.2f' % pred2019df["TotalAmount"].sum() )

There's a difference of over 2 million USD over this time frame, or a difference of about 13% compared to actuals. It might be interesting to see if we see any major differences when we slice by specific variables. 

### Prediction by Employee

Let's start with employee:

In [ ]:
pred2019df["Difference"] = pred2019df["TotalAmount"] - pred2019df["Prediction"]
pred2019df[["EmployeeID", "EmployeeName", "Difference"]].groupby(["EmployeeID", "EmployeeName"]).sum().style.format('${0:,.2f}')

What's interesting here is that four employees have major differences, while the rest are somewhat close. Villiers, Mowett, Harte, and Sweeting are all well above the expectation.  **THIS IS NOT PROOF OF FRAUD**.  There are a number of reasons why this may be normal--perhaps they picked up additional case loads or had to deal with some expensive situations.  But it **is** interesting and worth further investigation.

### Prediction by Expense Category

Similar to employee, we can look at expense categories to see where we were way off:

In [ ]:
pred2019df[["ExpenseCategoryID", "ExpenseCategory", "Difference"]] \
    .groupby(["ExpenseCategoryID", "ExpenseCategory"]) \
    .sum().sort_values("Difference", ascending=False) \
    .style.format('${0:,.2f}')

Our predictions for glass were way off, as well as mirrors, lights, and windshield wipers. The rest were off by some amount, but we were really bad at predicting glass and windshield prices.

### Prediction by Vendor

Finally, we'll look at our predictions by vendor to see if there are any clues there:

In [ ]:
pred2019df[["VendorID", "VendorName", "Difference"]] \
    .groupby(["VendorID", "VendorName"]) \
    .sum().sort_values("Difference", ascending=False) \
    .style.format('${0:,.2f}')

Well, that's weird. One vendor is way out in la-la land based on our regression analysis. This is still not evidence of fraud. For starters, our regression model only explained about 55-56% of all variance in prices. What that means is that the fact that reality didn't fit with our model isn't proof of anything. But it is a good time for us to circle back and think about other techniques for analyzing data.

## Trying Other Algorithms

We tried a simple linear regression model which proved to be reasonably interesting, but that is certainly not the limit of what regression models can give us. There are a variety of techniques within linear regression, as well as other interesting algorithms we can try. One such algorithm is <code>LightGBM</code>, initially developed by Microsoft Research. This is a model which builds out decision trees. A decision tree is essentially a series of if-else statements leading to an expected value.

Let's train on the pre-2019 data set. This will give us a point of comparison against the linear regression model.

In [ ]:
import lightgbm as lgb
import re

# Remove special characters from the dataframes.
reg1p_x_train_lgb = reg1p_x_train.rename(columns = lambda x:re.sub('[^A-Za-z0-9_]+', '', x))
reg1p_x_test_lgb = reg1p_x_test.rename(columns = lambda x:re.sub('[^A-Za-z0-9_]+', '', x))

lgb_train = lgb.Dataset(reg1p_x_train_lgb, reg1p_y_train)
lgb_test = lgb.Dataset(reg1p_x_test_lgb, reg1p_y_test)

# These are hyperparameters that we can tune.
params = {
    'boosting_type': 'gbdt',
    'objective': 'regression',
    'metric': {'l2', 'l1'},
    'num_leaves': 31,
    'learning_rate': 0.05,
    'feature_fraction': 0.9,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'verbose': 0
}

gbm = lgb.train(params, lgb_train, num_boost_round=100, valid_sets=lgb_test)

In [ ]:
gbm_pred = gbm.predict(reg1p_x_test_lgb, num_iteration=gbm.best_iteration)
print('Root mean squared error: %.2f' % root_mean_squared_error(reg1p_y_test, gbm_pred) )

The result is a model which is not really any better than our linear regression result. Given that, I would choose the linear regression model for a few reasons:

1. Linear regression is a fairly easy model to explain to people and is reasonably intuitive.
2. The gradient boosting model is not substantially better than linear regression at making predictions within the test set.
3. I do not have reason to doubt the following proposition: the actual relationship between my features (the inputs) and my label (amount spent) can be specified as a linear equation with an error term.

But we can see if LightGBM can help us predict something a bit more aggregated, such as monthly expenditures by fewer variables.

### Monthly Line Items

This time around, instead of breaking things out into a large number of variables, we will look solely at buses and their characteristics to see if our gradient boosting model does a good job. Also, we will change from total amount spent to number of line items to see if our models are better at estimating that.

In [ ]:
raw_data_bus = pd.read_sql("""SELECT
    c.CalendarYear,
    c.CalendarMonth,
    b.AverageRoadConditions,
    b.BusModelID,
    bm.BusModel,
    bm.ModelQuality,
    DATEDIFF(YEAR, b.DateFirstInService, li.LineItemDate) AS NumberOfYearsInService,
    SUM(li.Amount) AS TotalAmount,
    COUNT(*) AS NumberOfLineItems
FROM dbo.LineItem li
    INNER JOIN dbo.Calendar c
        ON li.LineItemDate = c.Date
    INNER JOIN dbo.Bus b
        ON li.BusID = b.BusID
    INNER JOIN dbo.BusModel bm
        ON b.BusModelID = bm.BusModelID
    INNER JOIN dbo.ExpenseCategory ec
        ON li.ExpenseCategoryID = ec.ExpenseCategoryID
    INNER JOIN dbo.Vendor v
        ON li.VendorID = v.VendorID
    INNER JOIN dbo.Employee e
        ON li.EmployeeID = e.EmployeeID
WHERE
    c.CalendarYear BETWEEN 2019 AND 2021
GROUP BY
    c.CalendarYear,
    c.CalendarMonth,
    b.AverageRoadConditions,
    b.BusModelID,
    bm.BusModel,
    bm.ModelQuality,
    DATEDIFF(YEAR, b.DateFirstInService, li.LineItemDate)
ORDER BY NEWID();""", engine)

It's easy enough to build out a linear regression model based on what we have so we'll start with that and then move to LightGBM afterward.

In [ ]:
# Build the independent variables
reg3_x = raw_data_bus[["CalendarYear", "CalendarMonth", "AverageRoadConditions", "BusModel", "ModelQuality", "NumberOfYearsInService"]]
reg3_x_onehot = pd.get_dummies(reg3_x)

# Capture the dependent variable
reg3_y = raw_data_bus["NumberOfLineItems"]

# Split out into training and test data sets
# We have just under 20,000 entries in total.
reg3_x_train = reg3_x_onehot[:-4000]
reg3_x_test = reg3_x_onehot[-4000:]

reg3_y_train = reg3_y[:-4000]
reg3_y_test = reg3_y[-4000:]

regr3 = LinearRegression()
model3 = regr3.fit(reg3_x_train, reg3_y_train)
sm_model3 = sm.OLS(reg3_y_train, reg3_x_train.astype(float)).fit()


In [ ]:
print('Training data set Coefficient of Determination', model3.score(reg3_x_train, reg3_y_train))
print('Test data set Coefficient of Determination', model3.score(reg3_x_test, reg3_y_test))

reg3_y_pred = regr3.predict(reg3_x_test)
print('Root mean squared error: %.2f' % root_mean_squared_error(reg3_y_test, reg3_y_pred) )

In [ ]:
# Remove special characters from the dataframes.
reg3_x_train_lgb = reg3_x_train.rename(columns = lambda x:re.sub('[^A-Za-z0-9_]+', '', x))
reg3_x_test_lgb = reg3_x_test.rename(columns = lambda x:re.sub('[^A-Za-z0-9_]+', '', x))

lgb_train3 = lgb.Dataset(reg3_x_train_lgb, reg3_y_train)
lgb_test3 = lgb.Dataset(reg3_x_test_lgb, reg3_y_test)

# These are hyperparameters that we can tune.
params = {
    'boosting_type': 'gbdt',
    'objective': 'regression',
    'metric': {'l2', 'l1'},
    'num_leaves': 31,
    'learning_rate': 0.05,
    'feature_fraction': 0.9,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'verbose': 0
}

gbm = lgb.train(params, lgb_train3, num_boost_round=100, valid_sets=lgb_test3)

In [ ]:
gbm_pred = gbm.predict(reg3_x_test_lgb, num_iteration=gbm.best_iteration)
print('Linear regression root mean squared error: %.2f' % root_mean_squared_error(reg3_y_test, reg3_y_pred) )
print('LightGBM root mean squared error: %.2f' % root_mean_squared_error(reg3_y_test, gbm_pred) )

print('Mean monthly expenditure amount: %.2f' % reg3_y.mean())

Now we do see an improvement: our gradient boosting model is just over half that of the linear regression model. The model still isn't perfect by any stretch, but if we're interested in the characteristics which explain bus sales, we can check those out.

**NOTE:**  In order to get the following to work on Windows, you need to install the graphviz binaries.  [Installation instructions are available on the graphviz website](https://forum.graphviz.org/t/new-simplified-installation-procedure-on-windows/224).

In [ ]:
import graphviz
from IPython.display import HTML
tree = lgb.create_tree_digraph(gbm, tree_index=1, orientation="vertical")
HTML(tree._repr_image_svg_xml())

The resulting model won't fit cleanly into a notebook, but you can open up the resulting SVG file in a web browser to see how it renders.